Instead of teaching the model to classify an image into a category, Im going to teach it to project an image into a 256-dimensional vector space. 

Goal: force similar paintings to have vectors that are close together, and dissimilar paintings to have vectors that are far apart.

Model choice:

Deep Metric Learning with Siamese Triplet Network

Why?
- Optimal for Content-Based Image Retrieval (CBIR), compared to standard approaches

- Siamese Network drops the classification head. Instead, acts as a spatial projector

- Triplet Loss
    - standard contrastive loss (comparing two images at a time), the network might just learn to output the exact same vector for every portrait, collapsing the space

    - Triplet Margin Loss prevents this by comparing three images simultaneously

    - forces the network to ensure the Positive is closer to the Anchor than the Negative ($n$) by a strictly defined mathematical margin ($\alpha$)

    $$\mathcal{L} = \max(d(a, p) - d(a, n) + \alpha, 0)$$


In [12]:
import pandas as pd
import numpy as np 
import os

import torch
import torch.nn as nn
import torch.nn.functional as F 
import torchvision
from PIL import Image
import random

### Dataset

In [13]:
data_df = pd.read_csv("./nga_dataset/nga_paintings_metadata.csv")
print(data_df.columns)

Index(['objectid', 'uuid_x', 'accessioned', 'accessionnum', 'locationid',
       'title', 'displaydate', 'beginyear', 'endyear', 'visualbrowsertimespan',
       'medium', 'dimensions', 'inscription', 'markings',
       'attributioninverted', 'attribution', 'provenancetext', 'creditline',
       'classification', 'subclassification', 'visualbrowserclassification',
       'parentid', 'isvirtual', 'departmentabbr', 'portfolio', 'series',
       'volume', 'watermarks', 'lastdetectedmodification', 'wikidataid',
       'customprinturl', 'uuid_y', 'iiifurl', 'iiifthumburl', 'viewtype',
       'sequence', 'width', 'height', 'maxpixels', 'openaccess', 'created',
       'modified', 'depictstmsobjectid', 'assistivetext'],
      dtype='str')


In [14]:
portrait_keywords = [
    'portrait', 'head', 'face', 'woman', 'man', 'boy', 'girl', 'madonna', 
    'saint', 'lady', 'gentleman', 'child', 'king', 'queen', 'duke', 
    'self-portrait', 'family', 'figure', 'nude', 'bather', 'peasant', 
    'soldier', 'christ', 'virgin', 'angel', 'apostle', 'pope', 'doge'
]

landscape_keywords = [
    'landscape', 'river', 'mountain', 'valley', 'view', 'scene', 'sea', 
    'beach', 'coast', 'forest', 'wood', 'tree', 'field', 'meadow', 
    'waterfall', 'lake', 'pond', 'harbor', 'bay', 'ruins', 'cityscape', 
    'bridge', 'canal', 'farm', 'sunset', 'sunrise'
]

still_life_keywords = [
    'still life', 'flowers', 'fruit', 'bowl', 'vase', 'basket', 'glass', 
    'table', 'plate', 'dish', 'bottle', 'cup', 'bread', 'cheese', 'wine', 
    'meat', 'fish', 'books', 'skull', 'bouquet', 'peaches', 'apples', 'grapes'
]

def pseudo_labeling(title):
    if not isinstance(title,str): return -1

    title = title.lower()

    if any(word in title for word in portrait_keywords): return 0
    elif any(word in title for word in landscape_keywords): return 1
    elif any(word in title for word in still_life_keywords): return 2
    else: return -1

data_df['pseudo_label'] = data_df['title'].apply(pseudo_labeling)

labeled_df = data_df[data_df['pseudo_label'] != -1].copy()
print(f"Paintings successfully pseudo-labeled: {len(labeled_df)}")

img_dir = "nga_dataset/images"
labeled_df['img_path'] = labeled_df['objectid'].apply(lambda x: os.path.join(img_dir,f"{x}.jpg"))

final_df = labeled_df[labeled_df['img_path'].apply(os.path.exists)].copy()
print(f"Final usable dataset size: {len(final_df)}")

print("\nClass Distribution (0: Portrait, 1: Landscape, 2: Still Life):")
print(final_df['pseudo_label'].value_counts())

Paintings successfully pseudo-labeled: 1670
Final usable dataset size: 1670

Class Distribution (0: Portrait, 1: Landscape, 2: Still Life):
pseudo_label
0    1065
1     400
2     205
Name: count, dtype: int64


In [15]:
class TripletDataset(torch.utils.data.Dataset):
    def __init__(self,df,transform=None):
        self.df = df
        self.transform = transform

        self.portraits = df[df['pseudo_label']==0]['img_path'].tolist()
        self.landscapes = df[df['pseudo_label']==1]['img_path'].tolist()
        self.still_life = df[df['pseudo_label']==2]['img_path'].tolist()

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        anchor_path = row['img_path']
        anchor_label = row['pseudo_label']
        
        positive_list,negative_list=[],[]

        if(anchor_label==0):
            positive_list = self.portraits
            negative_list = self.landscapes + self.still_life
        elif(anchor_label==1):
            positive_list = self.landscapes
            negative_list = self.portraits + self.still_life
        else:
            positive_list = self.still_life
            negative_list = self.portraits + self.landscapes
        
        pos_path = random.choice(positive_list)
        neg_path = random.choice(negative_list)
        while(pos_path==anchor_path): pos_path = random.choice(positive_list)

        anchor_img = Image.open(anchor_path).convert('RGB')
        pos_img = Image.open(pos_path).convert('RGB')
        neg_img = Image.open(neg_path).convert('RGB')

        anchor_img = self.transform(anchor_img)
        pos_img = self.transform(pos_img)
        neg_img = self.transform(neg_img)

        return anchor_img,pos_img,neg_img

In [16]:
train_transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize(256),
        torchvision.transforms.RandomCrop(224),
        torchvision.transforms.RandomHorizontalFlip(),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
)

dataset = TripletDataset(df=final_df,transform=train_transform)
anchor,pos,neg = dataset[0]
print(f"Anchor shape: {anchor.shape}")
print(f"Positive shape: {pos.shape}")
print(f"Negative shape: {neg.shape}")

Anchor shape: torch.Size([3, 224, 224])
Positive shape: torch.Size([3, 224, 224])
Negative shape: torch.Size([3, 224, 224])


### EmbeddingNet

Reusing the custom ResNet from Task1 

In [17]:
class ResidualBlock(nn.Module):
    def __init__(self, inchannel, outchannel, stride=1):
        super(ResidualBlock, self).__init__()
        self.left = nn.Sequential(
            nn.Conv2d(inchannel, outchannel, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(outchannel),
            nn.ReLU(inplace=True),
            nn.Conv2d(outchannel, outchannel, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(outchannel)
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or inchannel != outchannel:
            self.shortcut = nn.Sequential(
                nn.Conv2d(inchannel, outchannel, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(outchannel)
            )
            
    def forward(self, x):
        out = self.left(x)
        out = out + self.shortcut(x)
        out = F.relu(out)
        
        return out


class ResNet(nn.Module):
    def __init__(self, ResidualBlock):
        super(ResNet, self).__init__()
        self.inchannel = 64
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.layer1 = self.make_layer(ResidualBlock, 64, 2, stride=1)
        self.layer2 = self.make_layer(ResidualBlock, 128, 2, stride=2)
        self.layer3 = self.make_layer(ResidualBlock, 256, 2, stride=2)        
        self.layer4 = self.make_layer(ResidualBlock, 512, 2, stride=2)        
        
    def make_layer(self, block, channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.inchannel, channels, stride))
            self.inchannel = channels
        return nn.Sequential(*layers)
    
    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        return out


class EmbeddingNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(512, 256)
        self.AAP = nn.AdaptiveAvgPool2d((1,1))
        self.ResNet = ResNet(ResidualBlock)
    
    def forward(self,input):
        features = self.ResNet(input)
        features = self.AAP(features)
        features = features.flatten(start_dim=1)
        features = self.fc(features)
        features = F.normalize(features,p=2,dim=1)
        return features

### Loss

In [18]:
class GalleryDataset(torch.utils.data.Dataset):
    def __init__(self,df,transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        path = row['img_path']
        label = row['pseudo_label']

        img = Image.open(path).convert('RGB')
        img = self.transform(img)

        return (img,label)


eval_transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize(256),
        torchvision.transforms.CenterCrop(224),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
)

In [ ]:
model = EmbeddingNet().to('cuda')
dataloader = torch.utils.data.DataLoader(dataset,batch_size=8,shuffle=True)
optimizer = torch.optim.Adam(model.parameters(),lr=1e-4)
criterion = nn.TripletMarginLoss(margin=1.0)

num_epoch = 50
for epoch in range(num_epoch):
    model.train()
    running_loss = 0 
    running_pos = 0  
    running_neg = 0

    for i,batch in enumerate(dataloader):
        anchor_img,pos_img,neg_img = batch
        anchor_img = anchor_img.to('cuda')
        pos_img = pos_img.to('cuda')
        neg_img = neg_img.to('cuda')

        a_emb = model(anchor_img)
        p_emb = model(pos_img)
        n_emb = model(neg_img)

        pos_dist = F.pairwise_distance(a_emb,p_emb)
        neg_dist = F.pairwise_distance(a_emb,n_emb)

        running_pos += pos_dist.mean().item()
        running_neg += neg_dist.mean().item()

        loss = criterion(a_emb,p_emb,n_emb)
        running_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(dataloader):.4f}, \
          Pos Dist: {running_pos/len(dataloader):.4f}, Neg Dist: {running_neg/len(dataloader):.4f}")
    
    if((epoch+1)%10==0):
        gallery_emb, gallery_labels = [], []

        with torch.no_grad():
            model.eval()

            gd = GalleryDataset(final_df,transform=eval_transform)
            eval_loader = torch.utils.data.DataLoader(gd,batch_size=32,shuffle=False)

            for images,labels in eval_loader:
                images = images.to('cuda')

                embs = model(images)
                
                gallery_emb.append(embs.cpu())
                gallery_labels.append(labels)
            
            full_gallery = torch.cat(gallery_emb,dim=0)
            full_labels = torch.cat(gallery_labels,dim=0)

            # similarity matrix
            sim_matrix = torch.mm(full_gallery,full_gallery.t())
            sim_matrix.fill_diagonal_(-1.0)

            scores, indices = torch.topk(sim_matrix,k=5,dim=1)
            retrieved_labels = full_labels[indices]
            
            matches = retrieved_labels == full_labels.unsqueeze(1)
            hits = matches.any(dim=1)

            recall_at_5 = hits.float().mean().item()*100

            print(f"Recall@5: {recall_at_5:.2f}%")



### ViTEmbeddingNet

Model choice:

Vision Transformer (ViT-B/16) as a Siamese Feature Extractor (Transfer Learning)

Why?

- While CNNs (like ResNet) build understanding locally pixel-by-pixel, Vision Transformers use Self-Attention to evaluate the entire image globally from the very first layer. This is highly effective for art, where the "style" is often a holistic combination of color palette and global composition rather than just local edges.

- The [CLS] Token Projection: Instead of spatially pooling feature maps (like ResNet's AdaptiveAvgPool), the ViT slices the image into 16x16 patches and appends a dedicated Classification ([CLS]) token. Through self-attention, this single token aggregates the stylistic information of the entire painting.
    
    - We strip the standard 1000-class ImageNet head and project this [CLS] token through an nn.Linear(768, 256) layer, projecting the global attention context directly onto our spatial hypersphere.

- By initializing with ImageNet weights, the attention heads already possess a deep mathematical understanding of complex geometries, textures, and colors. Our Triplet Loss does not have to teach the model how to see; it only has to teach it how to cluster what it already sees.

- Triplet Margin Loss (The Constant): The geometric objective remains identical. By keeping the exact same loss function and $L_2$ normalization, we ensure a mathematically perfectly isolated A/B test between Convolutional filters and Self-Attention mechanisms.
    
    $$\mathcal{L} = \max(d(a, p) - d(a, n) + \alpha, 0)$$

In [20]:
class ViTEmbeddingNet(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.model = torchvision.models.vit_b_16(weights='DEFAULT')
        self.model.heads.head = nn.Linear(768,256)
    
    def forward(self,input):
        output = self.model(input)
        output = F.normalize(output,p=2,dim=1)
        return output

In [21]:
model = ViTEmbeddingNet().to('cuda')
dataloader = torch.utils.data.DataLoader(dataset,batch_size=8,shuffle=True)
optimizer = torch.optim.Adam(model.parameters(),lr=1e-5)
criterion = nn.TripletMarginLoss(margin=1.0)

num_epoch = 30
for epoch in range(num_epoch):
    model.train()
    running_loss = 0 
    running_pos = 0  
    running_neg = 0

    for i,batch in enumerate(dataloader):
        anchor_img,pos_img,neg_img = batch
        anchor_img = anchor_img.to('cuda')
        pos_img = pos_img.to('cuda')
        neg_img = neg_img.to('cuda')

        a_emb = model(anchor_img)
        p_emb = model(pos_img)
        n_emb = model(neg_img)

        pos_dist = F.pairwise_distance(a_emb,p_emb)
        neg_dist = F.pairwise_distance(a_emb,n_emb)

        running_pos += pos_dist.mean().item()
        running_neg += neg_dist.mean().item()

        loss = criterion(a_emb,p_emb,n_emb)
        running_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(dataloader):.4f}, \
          Pos Dist: {running_pos/len(dataloader):.4f}, Neg Dist: {running_neg/len(dataloader):.4f}")
    
    if((epoch+1)%10==0):
        gallery_emb, gallery_labels = [], []

        with torch.no_grad():
            model.eval()

            gd = GalleryDataset(final_df,transform=eval_transform)
            eval_loader = torch.utils.data.DataLoader(gd,batch_size=32,shuffle=False)

            for images,labels in eval_loader:
                images = images.to('cuda')

                embs = model(images)
                
                gallery_emb.append(embs.cpu())
                gallery_labels.append(labels)
            
            full_gallery = torch.cat(gallery_emb,dim=0)
            full_labels = torch.cat(gallery_labels,dim=0)

            # similarity matrix
            sim_matrix = torch.mm(full_gallery,full_gallery.t())
            sim_matrix.fill_diagonal_(-1.0)

            scores, indices = torch.topk(sim_matrix,k=5,dim=1)
            retrieved_labels = full_labels[indices]
            
            matches = retrieved_labels == full_labels.unsqueeze(1)
            hits = matches.any(dim=1)

            recall_at_5 = hits.float().mean().item()*100

            print(f"Recall@5: {recall_at_5:.2f}%")



Epoch 1, Loss: 0.5730,           Pos Dist: 0.9452, Neg Dist: 1.4532
Epoch 2, Loss: 0.3551,           Pos Dist: 0.7324, Neg Dist: 1.5892
Epoch 3, Loss: 0.2845,           Pos Dist: 0.6375, Neg Dist: 1.6002
Epoch 4, Loss: 0.2199,           Pos Dist: 0.5681, Neg Dist: 1.6091
Epoch 5, Loss: 0.1930,           Pos Dist: 0.5375, Neg Dist: 1.6376
Epoch 6, Loss: 0.1751,           Pos Dist: 0.5083, Neg Dist: 1.6443
Epoch 7, Loss: 0.1482,           Pos Dist: 0.4789, Neg Dist: 1.6362
Epoch 8, Loss: 0.1203,           Pos Dist: 0.4538, Neg Dist: 1.6512
Epoch 9, Loss: 0.1089,           Pos Dist: 0.4448, Neg Dist: 1.6681
Epoch 10, Loss: 0.0857,           Pos Dist: 0.4036, Neg Dist: 1.6924
Recall@5: 98.98%
Epoch 11, Loss: 0.0710,           Pos Dist: 0.4100, Neg Dist: 1.6746
Epoch 12, Loss: 0.0622,           Pos Dist: 0.3728, Neg Dist: 1.7095
Epoch 13, Loss: 0.0508,           Pos Dist: 0.3629, Neg Dist: 1.7256
Epoch 14, Loss: 0.0529,           Pos Dist: 0.3584, Neg Dist: 1.7005
Epoch 15, Loss: 0.0461,   